## Cell 1 — Library Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import warnings
import joblib
import os
warnings.filterwarnings("ignore")

from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from scipy.fft import fft, fftfreq

from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay, RocCurveDisplay, precision_recall_curve, average_precision_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
import sklearn

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print('imblearn not found. Install with: pip install imbalanced-learn')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

# GLOBAL STYLING
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
}),
PALETTE = {'Control': '#2ecc71', 'Cancer': '#e74c3c', 'Benign': '#e67e22'}
BINARY_PALETTE = {0: '#2ecc71', 1: '#e74c3c'}

# CONSTANTS
RANDOM_STATE = 42
N_FOLDS = 5
SAMPLING_RATE = 100
MEASUREMENT_DURATION = 60
N_POINTS = SAMPLING_RATE * MEASUREMENT_DURATION
N_SENSORS = 22
N_MOS = 10
N_EC = 8
N_PID = 1
N_QCM = 2

print('Libraries loaded')
print(f'Constants initialized: {N_FOLDS} folds, {N_POINTS} points')
print(f'scikit-learn: {sklearn.__version__}, imbalanced-learn: {"available" if SMOTE_AVAILABLE else "not available"}')
print(f'numpy: {np.__version__}, pandas: {pd.__version__}')


Note: you may need to restart the kernel to use updated packages.
Libraries loaded
Constants initialized: 5 folds, 6000 points
scikit-learn: 1.8.0, imbalanced-learn: available
numpy: 1.26.4, pandas: 3.0.2


## Cell 2 — Raw Sensor Calibration Functions

In [1]:
# ══════════════════════════════════════════════════════
# 2A. MOS SENSORS — Power-Law Model
# Rs/R0 = a · C^(-b)  →  C = (Rs/R0 / a)^(-1/b)

def mos_model(C, a, b):
    return a & np.power(C, -b)

def calibrate_mos(known_ppb, measured_Rs_R0):
    popt, pcov = curve_fit(
        mos_model, known_ppb, measured_Rs_R0,
        p0=[1.0, 0.3], bounds=([0, 0], [np.inf, np.inf]),
        maxfev=10000
    )
    a, b = popt
    perr = np.sqrt(np.diag(pcov))
    print(f'  MOS fit → a={a:.4f}±{perr[0]:.4f}, b={b:.4f}±{perr[1]:.4f}')
    return {'a': a, 'b': b, 'type': 'MOS'}

def mos_to_ppb(Rs_R0, params):
    a, b = params['a'], params['b']
    Rs_R0_clamped = np.clip(Rs_R0, 1e-6, None)
    return np.power(Rs_R0_clamped / a, -1.0 / b)

# ══════════════════════════════════════════════════════
# 2B. EC SENSORS — Linear Model
#     C (ppb) = (I - I_zero) / sensitivity

def calibrate_ec(known_ppb, measured_nA):
    lr = LinearRegression().fit(
        np.array(known_ppb).reshape(-1, 1),
        np.array(measured_nA)
    )
    sensitivity = lr.coef_[0]
    I_zero = lr.intercept_
    print(f'  EC fit  → sensitivity={sensitivity:.4f} nA/ppb, I_zero={I_zero:.2f} nA')
    return {'sensitivity': sensitivity, 'I_zero': I_zero, 'type': 'EC'}

def ec_to_ppb(I_nA, params):
    conc = (np.array(I_nA) - params['I_zero']) / params['sensitivity']
    return np.clip(conc, 0, None)

# ══════════════════════════════════════════════════════
# 2C. PID SENSOR — Linear + Response Factor Correction
#     C (ppb) = V_output / (RF_compound × Sensitivity_ref)

PID_RESPONSE_FACTORS = {
    'benzene':      0.53,
    'toluene':      0.54,
    'ethylbenzene': 0.52,
    'xylene':       0.52,
    'styrene':      0.40,
    'isobutylene':  1.00,
    'generic_btex': 0.52,
}

def calibrate_pid(known_ppb_isobutylene, measured_V):
    lr = LinearRegression().fit(
        np.array(known_ppb_isobutylene).reshape(-1, 1),
        np.array(measured_V)
    )
    sensitivity = lr.coef_[0]
    V_zero = lr.intercept_
    print(f'  PID fit → sensitivity={sensitivity:.6f} V/ppb, V_zero={V_zero:.4f} V')
    return {'sensitivity': sensitivity, 'V_zero': V_zero, 'type': 'PID'}

def pid_to_ppb(V_raw, params, compound='generic_btex'):
    RF = PID_RESPONSE_FACTORS.get(compound, 0.52)
    V_corrected = np.array(V_raw) - params['V_zero']
    conc = V_corrected / (RF * params['sensitivity'])
    return np.clip(conc, 0, None)

# ══════════════════════════════════════════════════════
# 2D. QCM SENSORS — Sauerbrey Equation
#     Δm = -Δf · A·√(ρq·μq) / (2f₀²)
#     C (ppb) via ideal gas law

QCM_CONSTANTS = {
    'f0':      10e6,     # Hz, resonant frequency
    'A':       1.54e-5,  # m², electrode area
    'rho_q':   2650,     # kg/m³, quartz density
    'mu_q':    2.947e10, # Pa, quartz shear modulus
    'V_cell':  50e-6,    # m³, sensor cell volume
    'T_K':     298,      # K, temperature (25°C)
    'P_Pa':    101325,   # Pa, atmospheric pressure
    'R_gas':   8.314,    # J/(mol·K)
}

VOC_MW = {
    'acetone':      0.05808,
    'isoprene':     0.06811,
    'ethanol':      0.04607,
    'pentane':      0.07215,
    'generic':      0.06000,
}

def qcm_to_ppb(delta_f, compound='generic', constants=None):
    if constants is None:
        constants = QCM_CONSTANTS
    
    f0    = constants['f0']
    A     = constants['A']
    rho_q = constants['rho_q']
    mu_q  = constants['mu_q']
    V_cell = constants['V_cell']
    T_K   = constants['T_K']
    P_Pa  = constants['P_Pa']
    R_gas = constants['R_gas']
    MW    = VOC_MW.get(compound, VOC_MW['generic'])
    
    # Sauerbrey: Mass-Change
    delta_m = (-np.array(delta_f) * A * np.sqrt(rho_q * mu_q)) / (2 * f0**2)
    delta_m = np.clip(delta_m, 0, None)  # mass can only increase on adsorption
    
    # Moles Adsorbed
    moles = delta_m / MW
    
    # Gas-phase concentration (mol/m³) → molar fraction → ppb
    # Using ideal gas: n/V = P/(RT) for total gas
    total_molar_density = P_Pa / (R_gas * T_K)  # mol/m³
    molar_fraction = (moles / V_cell) / total_molar_density
    ppb = molar_fraction * 1e9
    
    return ppb

# ══════════════════════════════════════════════════════
# 2E. Temperature & Humidity Compensation

def compensate_TRH(C_raw, T_C, RH_pct,
                    T_ref=25.0, RH_ref=50.0,
                    alpha_T=-0.008, alpha_RH=-0.003):
    delta_T  = np.array(T_C) - T_ref
    delta_RH = np.array(RH_pct) - RH_ref
    correction_factor = 1 + alpha_T * delta_T + alpha_RH * delta_RH
    correction_factor = np.clip(correction_factor, 0.5, 1.5)  # safety bounds
    return np.array(C_raw) / correction_factor

print('Calibration functions:')
print('   ├─ MOS  : power-law model (Rs/R0 = a·C^-b)')
print('   ├─ EC   : linear model (I = S·C + I_zero)')
print('   ├─ PID  : linear + response factor correction')
print('   ├─ QCM  : Sauerbrey equation + ideal gas law')
print('   └─ T/RH : linear compensation surface')


Calibration functions:
   ├─ MOS  : power-law model (Rs/R0 = a·C^-b)
   ├─ EC   : linear model (I = S·C + I_zero)
   ├─ PID  : linear + response factor correction
   ├─ QCM  : Sauerbrey equation + ideal gas law
   └─ T/RH : linear compensation surface


## Cell 3 — Simulated Calibration Data & Curve Fitting Demo

In [ ]:
import numpy as np

np.random.seed(RANDOM_STATE)

# 3A. MOS Calibration Example
print('─' * 55)
print('MOS Sensor Calibration (WO3 / Acetone)')
mos_cal_ppb   = np.array([10, 50, 100, 500, 1000, 2000, 5000])
true_a, true_b = 1.0, 0.35
mos_cal_Rs_R0 = mos_model(mos_cal_ppb, true_a, true_b) * (1 + 0.02 * np.random.randn(len(mos_cal_ppb)))
mos_params = calibrate_mos(mos_cal_ppb, mos_cal_Rs_R0)

# 3B. EC Calibration Example (e.g., formaldehyde sensor)
print('\nEC Sensor Calibration (Formaldehyde)')
ec_cal_ppb = np.array([0, 10, 50, 100, 200, 500])
ec_cal_nA  = 0.085 * ec_cal_ppb + 12.3 + 0.5 * np.random.randn(len(ec_cal_ppb))
ec_params  = calibrate_ec(ec_cal_ppb, ec_cal_nA)

# 3C. PID Calibration Example (isobutylene reference)
print('\nPID Sensor Calibration (Isobutylene Reference)')
pid_cal_ppb = np.array([0, 10, 50, 100, 500, 1000])
pid_cal_V   = 0.0042 * pid_cal_ppb + 0.01 + 0.001 * np.random.randn(len(pid_cal_ppb))
pid_params  = calibrate_pid(pid_cal_ppb, pid_cal_V)

# 3D. QCM Calibration — verify conversion
print('\nQCM Sensor (Sauerbrey verification)')
test_delta_f = np.array([-5, -25, -50, -100, -250])
qcm_ppb_out  = qcm_to_ppb(test_delta_f, compound='acetone')
for df_hz, c in zip(test_delta_f, qcm_ppb_out):
    print(f'  Δf = {df_hz:5.0f} Hz  →  {c:.2f} ppb acetone')

# 3E. Plot calibration curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MOS
C_range = np.linspace(10, 5000, 500)
axes[0].scatter(mos_cal_ppb, mos_cal_Rs_R0, color='#e74c3c', s=80, zorder=5, label='Cal points')
axes[0].plot(C_range, mos_model(C_range, mos_params['a'], mos_params['b']),
             color='#c0392b', linewidth=2, label=f"Fit: a={mos_params['a']:.3f}, b={mos_params['b']:.3f}")
axes[0].set_xlabel('Concentration (ppb)')
axes[0].set_ylabel('Rs / R0')
axes[0].set_title('MOS Calibration Curve\n(WO3 / Acetone)', fontweight='bold')
axes[0].set_xscale('log')
axes[0].legend(fontsize=9)

# EC
I_range = np.linspace(12, 60, 200)
axes[1].scatter(ec_cal_nA, ec_cal_ppb, color='#3498db', s=80, zorder=5, label='Cal points')
axes[1].plot(ec_params['sensitivity'] * np.array(ec_cal_ppb) + ec_params['I_zero'],
             ec_cal_ppb, color='#2980b9', linewidth=2,
             label=f"S={ec_params['sensitivity']:.4f} nA/ppb")
axes[1].set_xlabel('Current (nA)')
axes[1].set_ylabel('Concentration (ppb)')
axes[1].set_title('EC Calibration Curve\n(Formaldehyde)', fontweight='bold')
axes[1].legend(fontsize=9)

# QCM
df_range = np.linspace(-300, 0, 200)
axes[2].plot(df_range, qcm_to_ppb(df_range, compound='acetone'),
             color='#9b59b6', linewidth=2)
axes[2].scatter(test_delta_f, qcm_ppb_out, color='#8e44ad', s=80, zorder=5, label='Verification pts')
axes[2].set_xlabel('Frequency Shift Δf (Hz)')
axes[2].set_ylabel('Concentration (ppb)')
axes[2].set_title('QCM Calibration\n(Sauerbrey / Acetone)', fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('Sensor Calibration Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n Calibration parameters fitted and validated')

NameError: name 'RANDOM_STATE' is not defined

## Cell 4 - Apply Calibration: Raw Signals to Ppb Concentrations

In [4]:
# In real deployment: replace simulate_* functions with your
# actual STM32H7 ADC data acquisition output.

def simulate_breath_timeseries(sensor_type, baseline, peak_ppb, 
                                 n_points=N_POINTS, t_exposure=30,
                                 noise_level=0.02):
    """
    Simulate a realistic sensor response curve:
    - Phase 1: Zero baseline (0–5s)
    - Phase 2: Breath exposure with exponential rise (5–35s)
    - Phase 3: Recovery with exponential decay (35–60s)
    
    Returns: time array (s), raw signal array
    """
    t = np.linspace(0, MEASUREMENT_DURATION, n_points)
    signal = np.zeros(n_points)
    
    t_start, t_end = 5, 35  # exposure window
    tau_rise, tau_fall = 8, 15  # time constants (seconds)
    
    for i, ti in enumerate(t):
        if ti < t_start:
            signal[i] = baseline
        elif ti < t_end:
            rise = 1 - np.exp(-(ti - t_start) / tau_rise)
            signal[i] = baseline + peak_ppb * rise
        else:
            peak_reached = baseline + peak_ppb * (1 - np.exp(-(t_end - t_start) / tau_rise))
            signal[i] = baseline + (peak_reached - baseline) * np.exp(-(ti - t_end) / tau_fall)
    
    # Add realistic noise
    signal += noise_level * peak_ppb * np.random.randn(n_points)
    # Apply moving average smoothing (mimics STM32 pipeline)
    signal = np.convolve(signal, np.ones(10)/10, mode='same')
    return t, signal

# ── Generate one example measurement per sensor type ──────────────────
np.random.seed(42)
t, mos_raw_Rs   = simulate_breath_timeseries('MOS', baseline=10000, peak_ppb=800)
# Convert Rs to Rs/R0 (R0 = baseline resistance)
R0_mos = np.mean(mos_raw_Rs[:500])  # first 5s is zero-air baseline
mos_Rs_R0 = mos_raw_Rs / R0_mos
mos_ppb_ts = mos_to_ppb(mos_Rs_R0, mos_params)

_, ec_raw_nA    = simulate_breath_timeseries('EC', baseline=12.3, peak_ppb=50)
ec_ppb_ts       = ec_to_ppb(ec_raw_nA, ec_params)

_, pid_raw_V    = simulate_breath_timeseries('PID', baseline=0.01, peak_ppb=0.5)
pid_ppb_ts      = pid_to_ppb(pid_raw_V, pid_params, compound='toluene')

_, qcm_raw_df   = simulate_breath_timeseries('QCM', baseline=0, peak_ppb=-50)
qcm_ppb_ts      = qcm_to_ppb(qcm_raw_df, compound='acetone')

# ── Apply T/RH compensation ────────────────────────────────────────────
# Simulate SHT31 reading: 28°C, 65% RH (typical exhaled breath conditions)
T_measured  = 28.0
RH_measured = 65.0
mos_ppb_comp  = compensate_TRH(mos_ppb_ts, T_measured, RH_measured)
ec_ppb_comp   = compensate_TRH(ec_ppb_ts,  T_measured, RH_measured)
pid_ppb_comp  = compensate_TRH(pid_ppb_ts, T_measured, RH_measured)
qcm_ppb_comp  = compensate_TRH(qcm_ppb_ts, T_measured, RH_measured)

# ── Plot calibrated timeseries ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, ts, label, color, unit in zip(
    axes,
    [mos_ppb_comp, ec_ppb_comp, pid_ppb_comp, qcm_ppb_comp],
    ['MOS (WO₃ / Acetone)', 'EC (Formaldehyde)', 'PID (Toluene)', 'QCM (Acetone)'],
    ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'],
    ['ppb', 'ppb', 'ppb', 'ppb']
):
    ax.plot(t, ts, color=color, linewidth=1.2, alpha=0.9)
    ax.axvspan(5, 35, alpha=0.08, color=color, label='Breath exposure')
    ax.axvline(5, color=color, linestyle='--', alpha=0.6, linewidth=1)
    ax.axvline(35, color=color, linestyle='--', alpha=0.6, linewidth=1)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'Concentration ({unit})')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Calibrated Sensor Timeseries (T/RH Compensated)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('calibrated_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Raw signals converted to T/RH-compensated ppb concentrations')

NameError: name 'N_POINTS' is not defined

## Cell 5 - Feature Extraction from Timeseries

In [5]:
def extract_features_from_timeseries(signal, t,
                                      t_zero_end=5.0,
                                      t_exposure_start=5.0,
                                      t_exposure_end=35.0,
                                      t_recovery_end=60.0,
                                      n_fft_coeff=5):
    """
    Extract 6 core features from a single sensor timeseries.
    
    Args:
        signal             : 1D array of calibrated ppb values (6000 points)
        t                  : time array (seconds)
        t_zero_end         : end of zero-air baseline phase (s)
        t_exposure_start   : start of breath exposure (s)
        t_exposure_end     : end of breath exposure (s)
        t_recovery_end     : end of recovery phase (s)
        n_fft_coeff        : number of FFT coefficients to extract
    
    Returns:
        dict of 6 features
    """
    dt = t[1] - t[0]
    
    # Index masks
    mask_zero     = t < t_zero_end
    mask_exposure = (t >= t_exposure_start) & (t <= t_exposure_end)
    mask_recovery = (t > t_exposure_end) & (t <= t_recovery_end)
    
    # Baseline (zero-air signal)
    baseline = np.mean(signal[mask_zero]) if mask_zero.sum() > 0 else 0.0
    
    # Delta signal (baseline-subtracted)
    delta = signal - baseline
    delta_exposure = delta[mask_exposure]
    delta_recovery = delta[mask_recovery]
    
    # ── Feature 1: Peak Response ───────────────────────────────────────
    peak_response = float(np.max(np.abs(delta_exposure))) if len(delta_exposure) > 0 else 0.0
    
    # ── Feature 2: t90 — Response time to 90% of peak ─────────────────
    t90 = 0.0
    if peak_response > 0:
        threshold_90 = 0.90 * peak_response
        t_exp = t[mask_exposure]
        above = np.where(np.abs(delta_exposure) >= threshold_90)[0]
        t90 = float(t_exp[above[0]] - t_exposure_start) if len(above) > 0 else float(t_exposure_end - t_exposure_start)
    
    # ── Feature 3: t10 — Recovery time back to 10% of peak ────────────
    t10 = 0.0
    if peak_response > 0 and len(delta_recovery) > 0:
        threshold_10 = 0.10 * peak_response
        t_rec = t[mask_recovery]
        below = np.where(np.abs(delta_recovery) <= threshold_10)[0]
        t10 = float(t_rec[below[0]] - t_exposure_end) if len(below) > 0 else float(t_recovery_end - t_exposure_end)
    
    # ── Feature 4: Slope (rate of change in rising phase) ─────────────
    slope = 0.0
    if len(delta_exposure) > 1:
        # Linear fit to first 50% of exposure window (rising phase)
        n_half = len(delta_exposure) // 2
        t_rise = t[mask_exposure][:n_half]
        d_rise = delta_exposure[:n_half]
        if len(t_rise) > 2:
            coeffs = np.polyfit(t_rise, d_rise, 1)
            slope = float(coeffs[0])
    
    # ── Feature 5: AUC — Area Under Curve (exposure window) ───────────
    auc = 0.0
    if len(delta_exposure) > 1:
        t_exp = t[mask_exposure]
        auc = float(trapezoid(np.abs(delta_exposure), t_exp))
    
    # ── Feature 6: FFT dominant frequency coefficient ──────────────────
    # Extract top n_fft_coeff magnitude coefficients from exposure window
    fft_features = []
    if len(delta_exposure) > n_fft_coeff * 2:
        fft_vals = np.abs(fft(delta_exposure))[:len(delta_exposure)//2]
        # Sort by magnitude, take top n_fft_coeff
        top_idx = np.argsort(fft_vals)[-n_fft_coeff:]
        fft_features = fft_vals[np.sort(top_idx)].tolist()
    else:
        fft_features = [0.0] * n_fft_coeff
    
    # Return first FFT coefficient as representative scalar for basic feature set
    fft_dominant = float(fft_features[0]) if fft_features else 0.0
    
    return {
        'peak_response': peak_response,
        't90':           t90,
        't10':           t10,
        'slope':         slope,
        'auc':           auc,
        'fft_dominant':  fft_dominant,
    }


def extract_all_sensor_features(sensor_timeseries_dict, t):
    """
    Extract features from all 22 sensors.
    
    Args:
        sensor_timeseries_dict : {sensor_name: ppb_array}
        t                      : shared time array
    
    Returns:
        flat feature vector (132-dim), feature names list
    """
    all_features = {}
    feature_names = []
    
    for sensor_name, signal in sensor_timeseries_dict.items():
        feats = extract_features_from_timeseries(signal, t)
        for fname, fval in feats.items():
            key = f'{sensor_name}__{fname}'
            all_features[key] = fval
            feature_names.append(key)
    
    return np.array(list(all_features.values())), feature_names


# ── Demo: extract features from simulated sensor timeseries ───────────
sensor_timeseries = {
    'MOS1_WO3':    mos_ppb_comp,
    'EC1_CH2O':    ec_ppb_comp,
    'PID1_BTEX':   pid_ppb_comp,
    'QCM1_PDMS':   qcm_ppb_comp,
}
feat_vec, feat_names = extract_all_sensor_features(sensor_timeseries, t)

print(f'   Feature extraction complete')
print(f'   Sensors processed : {len(sensor_timeseries)}')
print(f'   Features extracted: {len(feat_vec)}')
print(f'\n   Feature vector preview:')
for name, val in zip(feat_names, feat_vec):
    print(f'   {name:<30} = {val:.4f}')

print(f'\n   For full 22-sensor array: 22 × 6 = 132 features')

NameError: name 'mos_ppb_comp' is not defined

## Cell 6 - Environment-Corrected Features

In [6]:
def extract_environment_features(breath_peaks, zero_peaks, ambient_peaks,
                                   epsilon=0.001):
    """
    Compute 5 environment-corrected features per sensor.
    
    Measurement protocol:
        1. Zero measurement   : sensor exposed to dry clean air (N2)
        2. Ambient measurement: sensor exposed to room air
        3. Breath measurement : sensor exposed to exhaled breath
    
    Args:
        breath_peaks  : array of peak responses during breath measurement
        zero_peaks    : array of peak responses during zero-air measurement
        ambient_peaks : array of peak responses during ambient measurement
        epsilon       : small constant to prevent division by zero
    
    Returns:
        dict of 5 environment-corrected features per sensor
    """
    breath  = np.array(breath_peaks,  dtype=float)
    zero    = np.array(zero_peaks,    dtype=float)
    ambient = np.array(ambient_peaks, dtype=float)
    
    # Feature 1: Breath - Zero (removes sensor drift/offset)
    breath_minus_zero = breath - zero
    
    # Feature 2: Ambient - Zero (background VOC level)
    ambient_minus_zero = ambient - zero
    
    # Feature 3: Breath - Ambient (net breath VOC above background)
    breath_minus_ambient = breath - ambient
    
    # Feature 4: Breath / Ambient ratio (normalises for environment variability)
    # epsilon prevents instability when ambient ≈ 0
    breath_over_ambient = breath / (ambient + epsilon)
    
    # Feature 5: Delta pre/post ambient (checks ambient drift during measurement)
    # In practice: compare ambient measured BEFORE vs AFTER breath measurement
    # Here we simulate a small ambient drift
    ambient_drift = ambient * 0.05  # placeholder — use real pre/post values
    delta_pre_post_ambient = ambient - ambient_drift
    
    return {
        'breath_minus_zero':         breath_minus_zero,
        'ambient_minus_zero':        ambient_minus_zero,
        'breath_minus_ambient':      breath_minus_ambient,
        'breath_over_ambient':       breath_over_ambient,
        'delta_pre_post_ambient':    delta_pre_post_ambient,
    }


# ── Demo with simulated 3-phase measurement ────────────────────────────
np.random.seed(RANDOM_STATE)
n_demo_sensors = 22

# Simulate peak responses for 22 sensors across 3 measurement phases
zero_peaks    = np.abs(np.random.normal(0.5, 0.1, n_demo_sensors))
ambient_peaks = np.abs(np.random.normal(2.0, 0.5, n_demo_sensors))
# Cancer patient: elevated breath VOC profile
breath_peaks_cancer  = ambient_peaks + np.abs(np.random.normal(8.0, 2.0, n_demo_sensors))
# Control patient: lower breath VOC profile
breath_peaks_control = ambient_peaks + np.abs(np.random.normal(2.5, 0.5, n_demo_sensors))

env_feats_cancer  = extract_environment_features(breath_peaks_cancer,  zero_peaks, ambient_peaks)
env_feats_control = extract_environment_features(breath_peaks_control, zero_peaks, ambient_peaks)

# ── Visualize environment-corrected features ──────────────────────────
sensor_labels = [f'S{i+1:02d}' for i in range(n_demo_sensors)]
feature_to_plot = 'breath_minus_ambient'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(n_demo_sensors)
axes[0].bar(x - 0.2, env_feats_control[feature_to_plot], 0.4,
            label='Control', color='#2ecc71', alpha=0.85)
axes[0].bar(x + 0.2, env_feats_cancer[feature_to_plot], 0.4,
            label='Cancer', color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(sensor_labels, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Breath - Ambient (ppb)')
axes[0].set_title('Breath–Ambient Response per Sensor', fontweight='bold')
axes[0].legend()

axes[1].bar(x - 0.2, env_feats_control['breath_over_ambient'], 0.4,
            label='Control', color='#2ecc71', alpha=0.85)
axes[1].bar(x + 0.2, env_feats_cancer['breath_over_ambient'], 0.4,
            label='Cancer', color='#e74c3c', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(sensor_labels, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Breath / Ambient Ratio')
axes[1].set_title('Breath/Ambient Ratio per Sensor', fontweight='bold')
axes[1].axhline(1.0, color='black', linestyle='--', alpha=0.5, label='Ambient baseline')
axes[1].legend()

plt.suptitle('Environment-Corrected Features (Cancer vs Control)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('environment_features.png', dpi=150, bbox_inches='tight')
plt.show()

total_env_features = n_demo_sensors * 5
total_features = n_demo_sensors * 6 + total_env_features
print(f'   Environment features computed')
print(f'   Core features        : {n_demo_sensors} sensors × 6 = {n_demo_sensors*6}')
print(f'   Environment features : {n_demo_sensors} sensors × 5 = {total_env_features}')
print(f'   Total feature vector : {total_features} dimensions')

NameError: name 'RANDOM_STATE' is not defined

## Cell 7 - Load Dataset & Binary Label Encoding

In [7]:
# Dataset: 427 patients × 27 VOC breath biomarkers
# Binary: Control=0, Cancer+Benign=1

DATA_PATH = 'LungCancer_cleaned.xlsx'
df_raw = pd.read_excel(DATA_PATH)

print('Dataset Overview')
print('=' * 50)
print(f'Shape           : {df_raw.shape}')
print(f'Patients        : {df_raw["PatientID"].nunique()}')
print(f'VOC Features    : {df_raw.shape[1] - 2}')
print(f'\nOriginal class distribution:')
for cls, cnt in df_raw['Class'].value_counts().items():
    pct = cnt / len(df_raw) * 100
    print(f'  {cls:<10} : {cnt:3d} ({pct:.1f}%)')

VOC_COLS = [c for c in df_raw.columns if c not in ['PatientID', 'Class']]

VOC_NAMES = {
    'CH2O':    'Formaldehyde',
    'C2H4O':   'Acetaldehyde',
    'C3H6O':   'Acetone',
    'C4H8O':   'Butanal',
    'C5H10O':  'Pentanal',
    'C6H12O':  'Hexanal',
    'C7H14O':  'Heptanal',
    'C8H16O':  'Octanal',
    'C9H18O':  'Nonanal',
    'C10H20O': 'Decanal',
    'C11H22O': 'Undecanal',
    'C12H24O': 'Dodecanal',
    'C13H26O': 'Tridecanal',
    'C4HO8O2': 'Methylglyoxal',
    'C2H4O2':  'Acetic Acid',
    'C3H4O':   'Acrolein',
    'C6H10O2': 'Caproic Acid',
    'C9H16O2': 'Azelaic Aldehyde',
    'C3H4O2':  'Acrylic Acid',
    'C4H6O2':  'Methacrylic Acid',
    'C4H6O':   'Methyl Vinyl Ketone',
    'C4H4O2':  'Succinaldehyde',
    'C5H8O':   'Cyclopentanone',
    'C7H6O':   'Benzaldehyde',
    'C7H11O':  'Cycloheptanone',
    'C13H22O': 'Geranylacetone',
    'C15H10O': 'Flavone',
}

# ── Binary label encoding ─────────────────────────────────────────────
# CLINICAL RATIONALE: Benign tumors still warrant positive screening
# to ensure follow-up; only confirmed healthy subjects are Control=0
df = df_raw.copy()
df['label'] = df['Class'].map({'Control': 0, 'Cancer': 1, 'Benign': 1})

print(f'\n🔵 Binary label encoding:')
print(f'   Control          → 0 (Negative / Healthy)')
print(f'   Cancer + Benign  → 1 (Positive / Cancer Screen)')
print(f'\n   Binary distribution:')
for lbl, cnt in df['label'].value_counts().items():
    name = 'Control' if lbl == 0 else 'Cancer+Benign'
    pct  = cnt / len(df) * 100
    print(f'   {name:<15} ({lbl}): {cnt:3d} ({pct:.1f}%)')

# ── Feature matrix ────────────────────────────────────────────────────
X = df[VOC_COLS].values.astype(float)
y = df['label'].values

# Handle any NaN/Inf (impute with column median)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

print(f'\nFeature matrix ready: X={X.shape}, y={y.shape}')
print(f'   NaN values remaining: {np.isnan(X).sum()}')

NameError: name 'pd' is not defined

## Cell 8 - Exploratory Data Analysis (EDA)

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── 8A. Class distribution ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
orig_counts = df_raw['Class'].value_counts()
bars = ax1.bar(orig_counts.index, orig_counts.values,
               color=[PALETTE[c] for c in orig_counts.index],
               edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, orig_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             str(val), ha='center', va='bottom', fontweight='bold')
ax1.set_title('Original Class Distribution', fontweight='bold')
ax1.set_ylabel('Count')

# ── 8B. Binary class distribution ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bin_counts = pd.Series(y).value_counts().sort_index()
labels_bin  = ['Control (0)', 'Cancer+Benign (1)']
colors_bin  = ['#2ecc71', '#e74c3c']
bars2 = ax2.bar(labels_bin, bin_counts.values, color=colors_bin,
                edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, bin_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             str(val), ha='center', va='bottom', fontweight='bold')
ax2.set_title('Binary Label Distribution', fontweight='bold')
ax2.set_ylabel('Count')

# ── 8C. Key VOC distributions (boxplot) ──────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
key_vocs = ['C3H6O', 'CH2O', 'C2H4O', 'C9H18O', 'C10H20O']  # clinically relevant
df_melt = df[key_vocs + ['Class']].melt(id_vars='Class', var_name='VOC', value_name='Conc')
voc_display = [VOC_NAMES.get(v, v) for v in key_vocs]
bp = ax3.boxplot(
    [df.loc[df['Class']=='Cancer', v].values for v in key_vocs],
    patch_artist=True, labels=[VOC_NAMES.get(v, v)[:8] for v in key_vocs]
)
for patch in bp['boxes']:
    patch.set_facecolor('#e74c3c')
    patch.set_alpha(0.7)
ax3.set_title('Key VOC Concentrations\n(Cancer patients)', fontweight='bold')
ax3.set_ylabel('Concentration (ppb)')
ax3.tick_params(axis='x', rotation=30)

# ── 8D. Correlation heatmap ───────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :])
corr_matrix = pd.DataFrame(X, columns=VOC_COLS).corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=ax4, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=VOC_COLS, yticklabels=VOC_COLS,
            cbar_kws={'shrink': 0.6})
ax4.set_title('VOC Feature Correlation Matrix', fontweight='bold', fontsize=12)
ax4.tick_params(axis='both', labelsize=7)

# ── 8E. Mean VOC profile: Control vs Cancer ───────────────────────────
ax5 = fig.add_subplot(gs[2, :])
mean_control = df[df['label']==0][VOC_COLS].mean()
mean_cancer  = df[df['label']==1][VOC_COLS].mean()
x_idx = np.arange(len(VOC_COLS))
ax5.plot(x_idx, mean_control.values, 'o-', color='#2ecc71', linewidth=2,
         markersize=5, label='Control (mean)', alpha=0.9)
ax5.plot(x_idx, mean_cancer.values,  's-', color='#e74c3c', linewidth=2,
         markersize=5, label='Cancer+Benign (mean)', alpha=0.9)
ax5.fill_between(x_idx,
                  df[df['label']==0][VOC_COLS].mean() - df[df['label']==0][VOC_COLS].std(),
                  df[df['label']==0][VOC_COLS].mean() + df[df['label']==0][VOC_COLS].std(),
                  alpha=0.15, color='#2ecc71')
ax5.fill_between(x_idx,
                  df[df['label']==1][VOC_COLS].mean() - df[df['label']==1][VOC_COLS].std(),
                  df[df['label']==1][VOC_COLS].mean() + df[df['label']==1][VOC_COLS].std(),
                  alpha=0.15, color='#e74c3c')
ax5.set_xticks(x_idx)
ax5.set_xticklabels([VOC_NAMES.get(v, v)[:10] for v in VOC_COLS],
                     rotation=45, ha='right', fontsize=8)
ax5.set_ylabel('Mean Concentration (ppb)')
ax5.set_title('Mean VOC Profile: Control vs Cancer+Benign (±1 SD)', fontweight='bold')
ax5.legend()

plt.suptitle('Exploratory Data Analysis — Lung Cancer VOC Dataset', 
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA complete')